# Logit Circuit Tracing

Trace the complete circuit from input to a specific logit: component
contributions, per-head breakdown, attribution paths, competing tokens,
and critical component identification.

## Why This Matters

Circuit logit tracing helps identify and characterize the specific subnetworks responsible for model behaviors. Finding circuits — the algorithms models actually implement — is the core goal of mechanistic interpretability.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_circuit_tracing import (
    trace_logit_to_components, per_head_logit_contribution,
    logit_attribution_path, competing_logit_analysis,
    critical_circuit_components,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Trace Logit to Components

How is a target token's logit built from each component?

In [ ]:
result = trace_logit_to_components(model, tokens, target_token=42, position=-1)
print(f"Target token: {result['target_token']}, actual logit: {result['actual_logit']:.4f}")
print(f"Traced total: {result['total_traced']:.4f}\n")
for c in result['components']:
    sign = '+' if c['logit_contribution'] > 0 else '-'
    print(f"  {c['name']:10s} ({c['type']:9s}): {sign}{abs(c['logit_contribution']):.4f}")

## Per-Head Logit Contribution

Break down one layer's attention logit by head.

In [ ]:
result = per_head_logit_contribution(model, tokens, target_token=42, layer=0)
print(f"Layer {result['layer']}, total attn logit: {result['total_attn_logit']:.4f}\n")
for h in result['per_head']:
    role = 'promotes' if h['promotes'] else 'SUPPRESSES'
    print(f"  Head {h['head']}: logit={h['logit_contribution']:+.4f}, "
          f"norm={h['output_norm']:.4f} [{role}]")

## Logit Attribution Path

Track how a logit builds up through layers.

In [ ]:
result = logit_attribution_path(model, tokens, target_token=42)
print(f"Final logit: {result['final_logit']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['attn_contribution']:+.4f}, "
          f"mlp={p['mlp_contribution']:+.4f}, cumulative={p['cumulative_logit']:+.4f}")

## Competing Logit Analysis

Which tokens compete for the highest logit?

In [ ]:
result = competing_logit_analysis(model, tokens, position=-1, top_k=5)
print(f"Winner: token {result['winner']}, margin: {result['margin']:.4f}\n")
for t in result['per_token']:
    print(f"  Token {t['token']:3d}: logit={t['final_logit']:+.4f}")
    for p in t['per_layer']:
        print(f"    L{p['layer']}: {p['contribution']:+.4f}")

## Critical Circuit Components

Which components are most important for a specific logit?

In [ ]:
result = critical_circuit_components(model, tokens, target_token=42)
print(f"Critical components: {result['n_critical']}/{len(result['components'])}\n")
for c in result['components'][:8]:
    tag = 'CRITICAL' if c['is_critical'] else ''
    print(f"  {c['name']:10s}: logit={c['logit_contribution']:+.4f}, "
          f"frac={c['fraction']:.2%} {tag}")